In [1]:
from pyspark.sql import SparkSession
import os

# Nạp đầy đủ Package cho Kafka, Iceberg, Nessie và S3 (MinIO)
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1,'
    'org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.3.1,'
    'org.projectnessie.nessie-integrations:nessie-spark-extensions-3.4_2.12:0.67.0,'
    'org.apache.hadoop:hadoop-aws:3.3.4 '  # THƯ VIỆN CÒN THIẾU Ở ĐÂY
    'pyspark-shell'
)

spark = SparkSession.builder \
    .appName("Fix-S3A-Error1") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "main") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://silver/") \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

# 2. Đọc bảng bằng Spark SQL
df = spark.sql("SELECT * FROM nessie.silver_db.survey_silver")

# 3. Chuyển đổi sang Pandas và hiển thị (đẹp hơn show())
# Lấy 10 dòng đầu tiên để xem nhanh
pdf = df.limit(5).toPandas()

In [2]:
pdf

,survey_id,age_group,race,is_hispanic,education,income,gender,state,use_frequency,household_size,amazon_device_count,processed_at
0,R_01vNIayewjIIKMF,35 - 44 years,Black or African American,Yes,Bachelor's degree,"$25,000 - $49,999",Male,New Jersey,Less than 5 times per month,1,1,2026-03-11 07:24:47.222204
1,R_037XK72IZBJyF69,55 - 64 years,White or Caucasian,No,Bachelor's degree,"$25,000 - $49,999",Female,Pennsylvania,Less than 5 times per month,2,1,2026-03-11 07:24:47.222204
2,R_038ZU6kfQ5f89fH,25 - 34 years,White or Caucasian,No,High school diploma or GED,"$25,000 - $49,999",Male,California,Less than 5 times per month,1,1,2026-03-11 07:24:47.222204
3,R_03aEbghUILs9NxD,35 - 44 years,White or Caucasian,No,Bachelor's degree,"$50,000 - $74,999",Male,Virginia,Less than 5 times per month,4,1,2026-03-11 07:24:47.222204
4,R_06RZP9pS7kONINr,65 and older,White or Caucasian,No,Bachelor's degree,"$75,000 - $99,999",Female,South Dakota,More than 10 times per month,2,1,2026-03-11 07:24:47.222204


In [3]:
import pandas as pd

# 1. Đọc toàn bộ dữ liệu cần thiết (hoặc một phần lớn để thống kê)
# Lưu ý: Nếu dữ liệu cực lớn, hãy cân nhắc dùng Spark functions để đếm trước khi toPandas
full_pdf = df.toPandas() 

print("--- THÔNG TIN CHI TIẾT DỮ LIỆU ---")

# A. In số dòng và số cột
rows, cols = full_pdf.shape
print(f"✅ Số lượng dòng: {rows}")
print(f"✅ Số lượng cột: {cols}")

# B. Kiểm tra dữ liệu thiếu (Missing values)
print("\n--- THỐNG KÊ DỮ LIỆU THIẾU ---")
missing_info = full_pdf.isnull().sum()
missing_percentage = (full_pdf.isnull().sum() / len(full_pdf)) * 100

# Tạo một DataFrame để hiển thị cho đẹp
missing_df = pd.DataFrame({
    'Cột': missing_info.index,
    'Số lượng Null': missing_info.values,
    'Tỷ lệ %': missing_percentage.values
})

print(missing_df.to_string(index=False))

# C. Hiển thị 5 dòng dữ liệu đầu tiên
print("\n--- DỮ LIỆU MẪU (TOP 5) ---")
display(full_pdf.head(5)) # Dùng display() nếu ở trong Notebook để có bảng đẹp

--- THÔNG TIN CHI TIẾT DỮ LIỆU ---
✅ Số lượng dòng: 5027
✅ Số lượng cột: 12

--- THỐNG KÊ DỮ LIỆU THIẾU ---
                Cột  Số lượng Null  Tỷ lệ %
          survey_id              0      0.0
          age_group              0      0.0
               race              0      0.0
        is_hispanic              0      0.0
          education              0      0.0
             income              0      0.0
             gender              0      0.0
              state              0      0.0
      use_frequency              0      0.0
     household_size              0      0.0
amazon_device_count              0      0.0
       processed_at              0      0.0

--- DỮ LIỆU MẪU (TOP 5) ---


,survey_id,age_group,race,is_hispanic,education,income,gender,state,use_frequency,household_size,amazon_device_count,processed_at
0,R_01vNIayewjIIKMF,35 - 44 years,Black or African American,Yes,Bachelor's degree,"$25,000 - $49,999",Male,New Jersey,Less than 5 times per month,1,1,2026-03-11 07:24:47.222204
1,R_037XK72IZBJyF69,55 - 64 years,White or Caucasian,No,Bachelor's degree,"$25,000 - $49,999",Female,Pennsylvania,Less than 5 times per month,2,1,2026-03-11 07:24:47.222204
2,R_038ZU6kfQ5f89fH,25 - 34 years,White or Caucasian,No,High school diploma or GED,"$25,000 - $49,999",Male,California,Less than 5 times per month,1,1,2026-03-11 07:24:47.222204
3,R_03aEbghUILs9NxD,35 - 44 years,White or Caucasian,No,Bachelor's degree,"$50,000 - $74,999",Male,Virginia,Less than 5 times per month,4,1,2026-03-11 07:24:47.222204
4,R_06RZP9pS7kONINr,65 and older,White or Caucasian,No,Bachelor's degree,"$75,000 - $99,999",Female,South Dakota,More than 10 times per month,2,1,2026-03-11 07:24:47.222204
